#### ***Rag Generation***
#### ***Rag Generation means generate the response from llm using user query and relevant document***

In [41]:
# Load environment variable
from dotenv import load_dotenv
load_dotenv()

True

In [42]:
###create a embedding by using langchain hugging face.

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [43]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000272D04E00B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000272D2D326C0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [44]:
## Load the vector store from Chroma
from langchain_chroma import Chroma
vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name = "kubernetes_rag",
    embedding_function=embedding_model
)

In [45]:
vectorstore._collection.count()

9507

In [46]:
### Similarity search
retriever = vectorstore.as_retriever(search_kwargs = {"k":3})

In [47]:
user_query ="What is a Kubernetes Deployment?"

In [48]:
### Test the retriever
retrieved_docs = retriever.invoke(user_query)

for doc in retrieved_docs:
    print("Page Content:",doc.page_content)
    print("#"*50)

Page Content: Kubernetes is a portable, extensible, open source platform for managing containerized
workloads and services, that facilitates both declarative configuration and automation. It has a
large, rapidly growing ecosystem. Kubernetes services, support, and tools are widely available.
The name Kubernetes originates from Greek, meaning helmsman or pilot. K8s as an
abbreviation results from counting the eight letters between the "K" and the "s". Google open-
sourced the Kubernetes project in 2014. Kubernetes combines over 15 years of Google's
experience running production workloads at scale with best-of-breed ideas and practices from
the community.
Going back in time
Let's take a look at why Kubernetes is so useful by going back in time.
Deployment evolution
Traditional deployment era: Early on, organizations ran applications on physical servers.
There was no way to define resource boundaries for applications in a physical server, and this
#########################################

In [49]:
##Design a Prompt
from langchain_core.prompts import ChatPromptTemplate
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_template("""
Answer the question using only the provided context.

If the answer is not present in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:
""")

In [50]:
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
rag_chain = ({
    "context":retriever | RunnableLambda(format_docs),
    "question":RunnablePassthrough()
}
|prompt
| llm 
| StrOutputParser()
)

In [51]:
user_query ="What is a Kubernetes Deployment?"

response = rag_chain.invoke(user_query)
print(response)

A Kubernetes Deployment is an API object that represents an application running on a cluster. It lets you declare the desired state of that application—such as how many replicas (pods) should be running—via a Deployment spec. The Kubernetes control plane reads this spec, creates the requested number of pod instances, and continuously monitors the status. If a pod fails or the actual state drifts from the spec, the system automatically corrects it (for example, by starting a replacement pod). In short, a Deployment is responsible for creating, scaling, and updating instances of your application to keep the cluster’s actual state aligned with the desired state you define.
